# 04 - Pressure ratings and rankings
Core rating (runs-equivalent per 100 balls vs an average player in the same state) = alpha (ordinary-ball skill) + delta (pressure deviation).
Style components (percentage points per 100 balls, each shrunk the same way):
  batters : dot-ball control (fewer dots than expected), boundary rate        bowlers : dot-ball rate, extras avoided
A style component only gets weight in the final score if it is repeatable (split-half rank correlation); otherwise its weight is cut.
Raw pressure-profile metrics (SR index, dot %, boundary:dot, chase-death rate delivery, dot chains, extras) are reported next to the rating.
Outputs: outputs/batter_ratings.csv, outputs/bowler_ratings.csv

In [1]:
import warnings
import numpy as np, pandas as pd
from pathlib import Path
from scipy.stats import spearmanr

warnings.filterwarnings('ignore')
pd.set_option('display.width', 250, 'display.max_columns', 60, 'display.max_rows', 80)
ROOT = Path.cwd() if (Path.cwd() / 'data').exists() else Path.cwd().parent
OUT = ROOT / 'outputs'
Bt = pd.read_parquet(OUT / 'batter_ball_values.parquet')
Bw = pd.read_parquet(OUT / 'bowler_ball_values.parquet')
b = pd.read_parquet(OUT / 'balls_pressure.parquet', columns=['striker_id', 'striker_name', 'bowler_id', 'bowler_name'])
names = pd.concat([b[['striker_id', 'striker_name']].set_axis(['pid', 'name'], axis=1),
                   b[['bowler_id', 'bowler_name']].set_axis(['pid', 'name'], axis=1)]).drop_duplicates('pid').set_index('pid').name

PRESSURE = 80                 # percentile (within tier) that counts as pressure
MIN_BALLS, MIN_INN = 60, 8    # eligibility for the top-10 lists (pressure balls / pressure innings with >=5 balls)
EXTRAS = {'bat': {'dot': 'vdot', 'bnd': 'vbnd'}, 'bowl': {'dot': 'vdot', 'ext': 'vext'}}
BASE_W = {'bat': {'pl': 0.60, 'dot': 0.15, 'bnd': 0.15, 'hit': 0.10},
          'bowl': {'pl': 0.60, 'dot': 0.15, 'ext': 0.15, 'hit': 0.10}}
GATE_AT = 0.25                # a style component earns its full weight only if its split-half rank correlation is >= this

## Empirical-Bayes helpers

In [2]:
def eb_center(m, n, sig2, min_n):
    """Shrink player means m (on n obs) toward the pooled mean. Returns shrunk mean, shrink factor B, tau (true spread)."""
    ok = n >= min_n
    tau2 = max(np.var(m[ok]) - np.mean(sig2 / n[ok]), 0.0)
    mu = np.average(m, weights=n)
    B = tau2 / (tau2 + sig2 / n)
    return mu + B * (m - mu), B, np.sqrt(tau2)


def comp_est(df, pid, col):
    """alpha (ordinary balls) + delta (pressure deviation) for one style component, shrunk; units = percentage points per 100 balls."""
    d = df[df[col].notna()].copy()
    d['wv'] = d[col] * d.w
    gp = d[d.hp].groupby(pid).agg(n_p=('w', 'sum'), s_p=('wv', 'sum'))
    go = d[~d.hp].groupby(pid).agg(n_o=('w', 'sum'), s_o=('wv', 'sum'))
    T = gp.join(go, how='left')
    T['n_o'] = T.n_o.fillna(0).clip(lower=1e-3)
    T['s_o'] = T.s_o.fillna(0)
    sig2 = d[col].var()
    mp, mo = T.s_p / T.n_p, T.s_o / T.n_o
    mo = mo.where(T.n_o > 1, np.average(mo, weights=T.n_o))
    a, _, _ = eb_center(mo.values, T.n_o.values, sig2, 100)
    dd = (mp - mo).values
    noise = sig2 / T.n_p.values + sig2 / T.n_o.values
    ok = (T.n_p.values >= 30) & (T.n_o.values >= 30)
    tau2 = max(np.var(dd[ok]) - np.mean(noise[ok]), 0.0)
    mu_d = np.average(dd, weights=1 / noise)
    Bd = tau2 / (tau2 + noise)
    dl = mu_d + Bd * (dd - mu_d)
    return pd.DataFrame({'alpha': a * 100, 'delta': dl * 100, 'pl': (a + dl) * 100}, index=T.index)


def fit_players(df, pid, pcol, extras, thr=PRESSURE, verbose=False):
    df = df[df[pcol].notna()].copy()
    df['hp'] = df[pcol] >= thr
    df['wr'], df['wo'] = df.vr * df.w, df.vo * df.w
    agg = lambda s: s.groupby(pid).agg(n=('w', 'sum'), raw=('w', 'size'), sr=('wr', 'sum'), so=('wo', 'sum'))
    P, O = agg(df[df.hp]), agg(df[~df.hp])
    T = P.join(O, lsuffix='_p', rsuffix='_o', how='left')
    T['n_o'] = T.n_o.fillna(0).clip(lower=1e-3)
    T[['sr_o', 'so_o', 'raw_o']] = T[['sr_o', 'so_o', 'raw_o']].fillna(0)
    alpha = delta = 0
    var_a = var_d = 0
    info = {}
    for k, (sp, so_) in {'r': ('sr_p', 'sr_o'), 'o': ('so_p', 'so_o')}.items():
        sig2 = df['v' + k].var()
        mp, mo = T[sp] / T.n_p, T[so_] / T.n_o
        mo = mo.where(T.n_o > 1, np.average(mo, weights=T.n_o))
        a, Ba, tau_a = eb_center(mo.values, T.n_o.values, sig2, 100)
        d = (mp - mo).values
        noise = sig2 / T.n_p.values + sig2 / T.n_o.values
        okk = (T.n_p.values >= 30) & (T.n_o.values >= 30)
        tau2_d = max(np.var(d[okk]) - np.mean(noise[okk]), 0.0)
        mu_d = np.average(d, weights=1 / noise)
        Bd = tau2_d / (tau2_d + noise)
        dd = mu_d + Bd * (d - mu_d)
        alpha, delta = alpha + a, delta + dd
        var_a = var_a + Ba * sig2 / T.n_o.values
        var_d = var_d + Bd * noise
        info[k] = (tau_a * 100, np.sqrt(tau2_d) * 100)
        T['raw_p_' + k] = mp * 100
    T['alpha'], T['delta'] = alpha, delta
    T['pl'] = alpha + delta
    T['pl_se'] = np.sqrt(var_a + var_d)
    T['delta_se'] = np.sqrt(var_d)
    for c in ['alpha', 'delta', 'pl', 'pl_se', 'delta_se']:
        T[c] *= 100
    T['m_p'] = (T.raw_p_r + T.raw_p_o)
    T['ci_lo'], T['ci_hi'] = T.pl - 1.96 * T.pl_se, T.pl + 1.96 * T.pl_se
    # style components
    for name, col in extras.items():
        est = comp_est(df, pid, col)
        T['pl_' + name] = est.pl.reindex(T.index)
        T['alpha_' + name] = est.alpha.reindex(T.index)
        T['delta_' + name] = est.delta.reindex(T.index)
    # consistency: share of pressure innings (>=5 pressure balls) with a positive total value
    h = df[df.hp].assign(wv=lambda x: (x.vr + x.vo) * x.w).groupby([pid, 'match_id', 'innings']).agg(n=('wv', 'size'), s=('wv', 'sum')).reset_index()
    h = h[h.n >= 5]
    hh = h.groupby(pid).agg(inn_p=('s', 'size'), hits=('s', lambda s: (s > 0).sum()))
    T = T.join(hh).fillna({'inn_p': 0, 'hits': 0})
    p0 = (h.s > 0).mean()
    T['hit'] = (T.hits + 6 * p0) / (T.inn_p + 6)
    mp_t, mo_t = (T.sr_p + T.so_p) / T.n_p, (T.sr_o + T.so_o) / T.n_o
    T['gap_raw'] = (mp_t - mo_t) * 100
    T['gap_z'] = (mp_t - mo_t) / np.sqrt(df.v.var() * (1 / T.n_p + 1 / T.n_o))
    T['weak_share'] = df[df.hp & df.match_tier.isin(['T20I_ASSOC', 'A_TEAM'])].groupby(pid).size().reindex(T.index).fillna(0) / T.raw_p
    if verbose:
        print('  true player spread (runs/100 balls)  runs: alpha %.1f, delta %.1f | wickets: alpha %.1f, delta %.1f' % (*info['r'], *info['o']))
    T['eligible'] = (T.raw_p >= MIN_BALLS) & (T.inn_p >= MIN_INN)
    return T


def composite(T, w, extras, calib=None, keep=None):
    """Weighted z-score. z-scales come from the `calib` players; scores are returned for the `keep` players."""
    c = T.eligible if calib is None else calib
    k = c if keep is None else keep
    z = lambda s: (s - s[c].mean()) / s[c].std()
    sc = w['pl'] * z(T.pl) + w.get('hit', 0) * z(T.hit)
    for name in extras:
        col = T['pl_' + name]
        sc = sc + w.get(name, 0) * z(col.fillna(col[c].mean()))
    return sc.where(k)

## Raw pressure-profile metrics (what a scout reads next to the rating)
Batters: SR vs expected, dot %, boundary:dot ratio (smoothed), rate delivery vs the required rate in chase death overs,
dots while the required rate is rising, and how often he breaks a dot chain. Bowlers: dot %, over-level dot %, extras.

In [3]:
def raw_bat(df, thr=PRESSURE):
    hp = df[df.p_bat >= thr]
    g = hp.groupby('striker_id')
    R = pd.DataFrame({'strike_rate_p': g.runs.mean() * 100, 'strike_rate_exp_p': g.exp_runs.mean() * 100, 'dot_p': g.bdot.mean() * 100,
                      'dot_exp_p': g.exp_dot.mean() * 100, 'bnd_p': g.is_bnd.mean() * 100})
    R['sr_idx'] = R.strike_rate_p / R.strike_rate_exp_p * 100
    R['bd_ratio_p'] = (g.is_bnd.sum() + 1) / (g.bdot.sum() + 1)
    R['bd_ratio_exp_p'] = (g.exp_bnd.sum() + 1) / (g.exp_dot.sum() + 1)
    cd = df[(df.innings == 2) & (df.overid >= 16) & df.rrr.between(0, 20) & df.wp.between(0.05, 0.95)]
    gg = cd.groupby('striker_id')
    R['rdi_death'] = gg.runs.sum() / (gg.rrr.sum() / 6)        # > 1: scores faster than the required rate asks
    R['rdi_n'] = gg.size()
    rs = df[(df.innings == 2) & (df.rrr_trend > 0) & df.wp.between(0.05, 0.95)]
    gr = rs.groupby('striker_id')
    R['dot_rise'] = gr.bdot.mean() * 100
    R['dot_rise_exp'] = gr.exp_dot.mean() * 100
    R['rise_n'] = gr.size()
    gc = rs[rs.dot_streak >= 2].groupby('striker_id')
    R['chain_break'] = (1 - gc.bdot.mean()) * 100               # % of balls scored off after 2+ dots, required rate rising
    R['chain_break_exp'] = (1 - gc.exp_dot.mean()) * 100
    R['chain_n'] = gc.size()
    return R


def raw_bowl(df, thr=PRESSURE):
    hp = df[df.p_bowl >= thr]
    g = hp.groupby('bowler_id')
    leg = hp[hp.legal]
    gl = leg.groupby('bowler_id')
    R = pd.DataFrame({'dot_pct_p': gl.bowl_dot.mean() * 100, 'dot_exp_p': gl.exp_bdot.mean() * 100,
                      'econ_p': g.bowler_runs.sum() / gl.size() * 6})
    R['ext_per_over_p'] = g.bowl_extra.sum() / gl.size() * 6
    R['ext_exp_over_p'] = g.exp_ext.sum() / gl.size() * 6
    R['ext_runs_p'] = hp[hp.bowl_extra].groupby('bowler_id').bowler_runs.sum()
    ov = df[df.innings.isin([1, 2])].groupby(['bowler_id', 'match_id', 'innings', 'overid']).agg(
        p=('p_bowl', 'mean'), legal=('legal', 'sum'), dots=('bowl_dot', 'sum')).reset_index()
    ov = ov[(ov.p >= thr) & (ov.legal >= 4)]
    ov['dp'] = ov.dots / ov.legal
    R['over_dot_pct_p'] = ov.groupby('bowler_id').dp.mean() * 100
    R['p_overs'] = ov.groupby('bowler_id').size()
    return R

## Reliability first: which metrics are repeatable? (odd vs even matches; players with >=30 pressure balls in both halves)

In [4]:
def rel(df, pid, pcol, extras):
    Ta = fit_players(df[df.match_id % 2 == 0], pid, pcol, extras)
    Tb = fit_players(df[df.match_id % 2 == 1], pid, pcol, extras)
    j = Ta.join(Tb, lsuffix='_a', rsuffix='_b', how='inner')
    j = j[(j.raw_p_a >= 30) & (j.raw_p_b >= 30)]
    cols = ['m_p', 'pl', 'alpha', 'gap_raw', 'hit'] + ['pl_' + k for k in extras]
    r = {c: round(spearmanr(j[c + '_a'], j[c + '_b'], nan_policy='omit')[0], 2) for c in cols}
    r['n_players'] = len(j)
    return r


def gate(base, r, extras):
    raw = {'pl': base['pl'], 'hit': base['hit']}
    for k in extras:
        raw[k] = base[k] * float(np.clip(r['pl_' + k] / GATE_AT, 0, 1))
    tot = sum(raw.values())
    return {k: v / tot for k, v in raw.items()}


REL = {'bat': rel(Bt, 'striker_id', 'p_bat', EXTRAS['bat']), 'bowl': rel(Bw, 'bowler_id', 'p_bowl', EXTRAS['bowl'])}
W = {r: gate(BASE_W[r], REL[r], EXTRAS[r]) for r in REL}
print('Split-half rank correlation (m_p = raw pressure mean, what a naive metric uses)')
print('batters :', REL['bat']); print('bowlers :', REL['bowl'])
print('\nFinal-score weights after reliability gating (a metric counts only as much as it repeats)')
print('batters :', {k: round(v, 2) for k, v in W['bat'].items()}); print('bowlers :', {k: round(v, 2) for k, v in W['bowl'].items()})

Split-half rank correlation (m_p = raw pressure mean, what a naive metric uses)
batters : {'m_p': np.float64(-0.14), 'pl': np.float64(0.39), 'alpha': np.float64(0.38), 'gap_raw': np.float64(-0.24), 'hit': np.float64(0.0), 'pl_dot': np.float64(0.51), 'pl_bnd': np.float64(0.55), 'n_players': 143}
bowlers : {'m_p': np.float64(0.17), 'pl': np.float64(0.4), 'alpha': np.float64(0.41), 'gap_raw': np.float64(0.01), 'hit': np.float64(0.2), 'pl_dot': np.float64(0.37), 'pl_ext': np.float64(0.42), 'n_players': 144}

Final-score weights after reliability gating (a metric counts only as much as it repeats)
batters : {'pl': 0.6, 'hit': 0.1, 'dot': 0.15, 'bnd': 0.15}
bowlers : {'pl': 0.6, 'hit': 0.1, 'dot': 0.15, 'ext': 0.15}


## Ratings and scores

In [5]:
def finish(T, role):
    T['score'] = composite(T, W[role], EXTRAS[role])
    T['rank'] = T.score.rank(ascending=False)
    T['pressure_type'] = np.select([T.gap_z >= 1.5, T.gap_z <= -1.5], ['Thrives', 'Dips'], 'Neutral')   # indicative only
    T.insert(0, 'name', T.index.map(names))
    return T.sort_values('score', ascending=False)


print('\nBATTERS'); BAT = finish(fit_players(Bt, 'striker_id', 'p_bat', EXTRAS['bat'], verbose=True), 'bat')
print('BOWLERS'); BOWL = finish(fit_players(Bw, 'bowler_id', 'p_bowl', EXTRAS['bowl'], verbose=True), 'bowl')
BAT = BAT.join(raw_bat(Bt))
BOWL = BOWL.join(raw_bowl(Bw))
print('eligible batters:', int(BAT.eligible.sum()), '| eligible bowlers:', int(BOWL.eligible.sum()))

SB = ['name', 'raw_p', 'inn_p', 'alpha', 'pl', 'pl_dot', 'pl_bnd', 'ci_lo', 'ci_hi', 'sr_idx', 'dot_p', 'bd_ratio_p', 'score']
SW = ['name', 'raw_p', 'inn_p', 'alpha', 'pl', 'pl_dot', 'pl_ext', 'ci_lo', 'ci_hi', 'dot_pct_p', 'ext_per_over_p', 'score']
print('\nTOP 15 BATTERS UNDER PRESSURE (pl = runs-equivalent per 100 balls; pl_dot / pl_bnd = percentage points per 100 balls)')
print(BAT[BAT.eligible].head(15)[SB].round(2).to_string())
print('\nTOP 15 BOWLERS UNDER PRESSURE')
print(BOWL[BOWL.eligible].head(15)[SW].round(2).to_string())


BATTERS
  true player spread (runs/100 balls)  runs: alpha 11.8, delta 0.0 | wickets: alpha 0.0, delta 0.0
BOWLERS
  true player spread (runs/100 balls)  runs: alpha 7.4, delta 12.7 | wickets: alpha 1.2, delta 0.0
eligible batters: 124 | eligible bowlers: 135

TOP 15 BATTERS UNDER PRESSURE (pl = runs-equivalent per 100 balls; pl_dot / pl_bnd = percentage points per 100 balls)
                        name  raw_p  inn_p  alpha     pl  pl_dot  pl_bnd  ci_lo  ci_hi  sr_idx  dot_p  bd_ratio_p  score
striker_id                                                                                                              
6281         Abhishek Sharma    245   17.0  31.33  34.09    4.46    6.60  26.91  41.28  127.22  34.29        0.86   2.92
25669          Priyansh Arya     62    8.0  25.37  28.13    2.46    4.89  14.01  42.26  127.68  38.71        0.84   2.19
6396               Tim David    277   22.0  21.86  24.63    1.57    5.01  14.48  34.78  120.36  30.69        0.77   1.91
5553           

## Do the new metrics change the answer? Overlap of the top 10 with the core-only score, and sensitivity to other choices

In [6]:
for lab, role, T, df, pid, pc in [('BATTERS', 'bat', BAT, Bt, 'striker_id', 'p_bat'), ('BOWLERS', 'bowl', BOWL, Bw, 'bowler_id', 'p_bowl')]:
    ex = EXTRAS[role]
    cur = list(T[T.eligible].sort_values('score', ascending=False).head(10).index)
    core = list(composite(T, {'pl': 0.9, 'hit': 0.1}, ex).sort_values(ascending=False).head(10).index)
    print(lab, '- overlap with CORE-ONLY top 10 (runs + wickets only, no style metrics):', len(set(cur) & set(core)), '/10')
    for k in ex:
        only = composite(T, {'pl': 0.0, 'hit': 0.0, k: 1.0}, ex)
        print('  top 10 by %s alone shares %d/10 with the chosen top 10' % (k, len(set(cur) & set(only.sort_values(ascending=False).head(10).index))))
    for thr in (70, 90):
        T2 = fit_players(df, pid, pc, ex, thr=thr)
        print('  pressure threshold', thr, '->', len(set(cur) & set(composite(T2, W[role], ex).sort_values(ascending=False).head(10).index)), '/10')
    T3 = fit_players(df.assign(w=1.0), pid, pc, ex)
    print('  all tiers weighted equally ->', len(set(cur) & set(composite(T3, W[role], ex).sort_values(ascending=False).head(10).index)), '/10')
    T4 = fit_players(df[~df.match_tier.isin(['T20I_ASSOC', 'A_TEAM'])], pid, pc, ex)
    print('  IPL + full-member T20I only ->', len(set(cur) & set(composite(T4, W[role], ex).sort_values(ascending=False).head(10).index)), '/10')
    e = T.eligible & (T.raw_p >= 100) & (T.inn_p >= 12)
    print('  stricter eligibility (100 balls, 12 inns; %d players) ->' % e.sum(), len(set(cur) & set(composite(T, W[role], ex, calib=e).sort_values(ascending=False).head(10).index)), '/10')

BATTERS - overlap with CORE-ONLY top 10 (runs + wickets only, no style metrics): 9 /10
  top 10 by dot alone shares 2/10 with the chosen top 10
  top 10 by bnd alone shares 8/10 with the chosen top 10
  pressure threshold 70 -> 8 /10
  pressure threshold 90 -> 3 /10
  all tiers weighted equally -> 10 /10
  IPL + full-member T20I only -> 9 /10
  stricter eligibility (100 balls, 12 inns; 66 players) -> 7 /10
BOWLERS - overlap with CORE-ONLY top 10 (runs + wickets only, no style metrics): 7 /10
  top 10 by dot alone shares 3/10 with the chosen top 10
  top 10 by ext alone shares 3/10 with the chosen top 10
  pressure threshold 70 -> 7 /10
  pressure threshold 90 -> 5 /10
  all tiers weighted equally -> 9 /10
  IPL + full-member T20I only -> 8 /10
  stricter eligibility (100 balls, 12 inns; 87 players) -> 9 /10


## Consensus ranking: average the score over 12 reasonable specifications
pressure threshold {70, 80, 90} x weak-tier weight {0.5, 1.0} x with / without the consistency term

In [7]:
def consensus(df, role, pid, pc, base, min_balls=30):
    ex = EXTRAS[role]
    el = base.eligible                       # z-scores are calibrated on players meeting the full threshold ...
    ok = base.raw_p >= min_balls             # ... but every player with >= 30 pressure balls gets a score (thin samples are flagged)
    zs = []
    weak = df.match_tier.isin(['T20I_ASSOC', 'A_TEAM'])
    nohit = {k: (0.0 if k == 'hit' else v) for k, v in W[role].items()}
    s = sum(nohit.values())
    nohit = {k: v / s for k, v in nohit.items()}
    for thr in (70, 80, 90):
        for tw in (0.5, 1.0):
            T = fit_players(df.assign(w=df.w.where(~weak, tw)), pid, pc, ex, thr=thr).reindex(base.index)
            for w in (W[role], nohit):
                zs.append(composite(T, w, ex, calib=el, keep=ok))
    return pd.concat(zs, axis=1).mean(axis=1)


for lab, role, T, df, pid, pc in [('BAT', 'bat', BAT, Bt, 'striker_id', 'p_bat'), ('BOWL', 'bowl', BOWL, Bw, 'bowler_id', 'p_bowl')]:
    T['consensus'] = consensus(df, role, pid, pc, T)
    T['final_rank'] = T.consensus.where(T.eligible).rank(ascending=False)
    print(lab, 'overlap of consensus top 10 with base-spec top 10:', len(set(T.nsmallest(10, 'final_rank').index) & set(T.nsmallest(10, 'rank').index)), '/10')

FIN = ['name', 'final_rank', 'rank', 'raw_p', 'inn_p', 'alpha', 'pl', 'ci_lo', 'ci_hi', 'pl_dot', 'consensus', 'pressure_type']
print('\nFINAL TOP 10 BATTERS'); print(BAT.nsmallest(10, 'final_rank')[FIN + ['pl_bnd', 'sr_idx', 'dot_p', 'rdi_death']].round(2).to_string())
print('\nFINAL TOP 10 BOWLERS'); print(BOWL.nsmallest(10, 'final_rank')[FIN + ['pl_ext', 'dot_pct_p', 'ext_per_over_p']].round(2).to_string())

BAT overlap of consensus top 10 with base-spec top 10: 10 /10
BOWL overlap of consensus top 10 with base-spec top 10: 7 /10

FINAL TOP 10 BATTERS
                        name  final_rank  rank  raw_p  inn_p  alpha     pl  ci_lo  ci_hi  pl_dot  consensus pressure_type  pl_bnd  sr_idx  dot_p  rdi_death
striker_id                                                                                                                                                 
6281         Abhishek Sharma         1.0   1.0    245   17.0  31.33  34.09  26.91  41.28    4.46       3.08       Neutral    6.60  127.22  34.29        NaN
25669          Priyansh Arya         2.0   2.0     62    8.0  25.37  28.13  14.01  42.26    2.46       2.20       Neutral    4.89  127.68  38.71        NaN
6396               Tim David         3.0   3.0    277   22.0  21.86  24.63  14.48  34.78    1.57       1.98       Neutral    5.01  120.36  30.69       0.96
1699        Suryakumar Yadav         4.0   5.0    382   25.0  19.53  22.30

## Pressure gap vs own baseline (raw, indicative only: the gap itself has low reliability)

In [8]:
for lab, T in [('BATTERS', BAT), ('BOWLERS', BOWL)]:
    e = T[T.eligible]
    print('\n%s: pressure_type counts' % lab, e.pressure_type.value_counts().to_dict())
    print(' biggest risers:'); print(e.sort_values('gap_z', ascending=False).head(4)[['name', 'alpha', 'gap_raw', 'gap_z', 'raw_p']].round(2).to_string())
    print(' biggest dippers:'); print(e.sort_values('gap_z').head(4)[['name', 'alpha', 'gap_raw', 'gap_z', 'raw_p']].round(2).to_string())

BAT.to_csv(OUT / 'batter_ratings.csv')
BOWL.to_csv(OUT / 'bowler_ratings.csv')
print('\nsaved ratings')


BATTERS: pressure_type counts {'Neutral': 112, 'Thrives': 8, 'Dips': 4}
 biggest risers:
                        name  alpha  gap_raw  gap_z  raw_p
striker_id                                                
4260        Heinrich Klaasen   8.04    53.58   3.20    287
109          Wriddhiman Saha -18.69    70.73   2.58    102
6460            Ayush Badoni -10.23    42.96   2.28    265
1107           Mohammad Nabi  -6.16    41.82   2.10    314
 biggest dippers:
                       name  alpha  gap_raw  gap_z  raw_p
striker_id                                               
2101        Quinton de Kock   3.08   -62.44  -2.87    155
2367            Travis Head  17.90   -44.20  -2.05    169
2416            Kyle Mayers   4.65   -45.54  -1.78    118
3639           Brandon King  -3.68   -41.09  -1.57    103

BOWLERS: pressure_type counts {'Neutral': 120, 'Dips': 10, 'Thrives': 5}
 biggest risers:
                          name  alpha  gap_raw  gap_z  raw_p
bowler_id                             